In [15]:
import torch
import torch.nn as torch_nn
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm 

import core.utils as utils
import core.plots as plots
import core.models as models

if torch.cuda.is_available():
    dev = "cuda:0"
    torch.set_default_device(dev)
    print(f"Using device: {dev}")
  
else:
    dev = "cpu"
    print(f"Using device: {dev}")

train_7x14 = True  # Set to True to train NAF for 7x7 -> 14x14, False for 14x14 -> 28x28

import sys
import os

# 1. On ajoute le dossier 'naf' tout en haut de la liste de recherche de Python
naf_path = os.path.abspath('naf')
if naf_path not in sys.path:
    sys.path.insert(0, naf_path)
from naf.src.model import NAF 

Using device: cuda:0


In [ ]:

# FONCTIONS UTILITAIRES 

def reshape_dofa(raw_features):
    """Transforme la sortie de DOFA en grille 4D spatiale : (Batch, Canaux, H, W)."""
    
    if len(raw_features.shape) == 3 and raw_features.shape[1] == 14 and raw_features.shape[2] == 14:
        return raw_features.unsqueeze(0) # Ajoute la dimension Batch -> (1, 1024, 14, 14)
        
    elif len(raw_features.shape) == 3:
        B, N, C = raw_features.shape
        # Si on a 197 tokens (14x14 + 1 class token), on enlève le premier
        patch_tokens = raw_features[:, 1:, :] if N == 197 else raw_features
        return patch_tokens.permute(0, 2, 1).reshape(B, C, 14, 14)
        
    return raw_features


def get_target_hr_features(dofa_model, hsi_image, wavelengths):
    """
    Découpe l'image HSI (128x128) en 4 tuiles (64x64), 
    les agrandit en 224x224, extrait les features DOFA, puis les réassemble.
    """
    B, C, H, W = hsi_image.shape
    
    tl = hsi_image[:, :, 0:64, 0:64]
    tr = hsi_image[:, :, 0:64, 64:128]
    bl = hsi_image[:, :, 64:128, 0:64]
    br = hsi_image[:, :, 64:128, 64:128]
    
    tuiles = [tl, tr, bl, br]
    tuiles_224 = [F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False) for t in tuiles]
    
    with torch.no_grad():
        # Extraction et reshape immédiat pour chaque quadrant
        feat_tl = reshape_dofa(models.extract_features(dofa_model, tuiles_224[0], wavelengths))
        feat_tr = reshape_dofa(models.extract_features(dofa_model, tuiles_224[1], wavelengths))
        feat_bl = reshape_dofa(models.extract_features(dofa_model, tuiles_224[2], wavelengths))
        feat_br = reshape_dofa(models.extract_features(dofa_model, tuiles_224[3], wavelengths))
        
    top = torch.cat([feat_tl, feat_tr], dim=3)    # -> (B, 1024, 14, 28)
    bottom = torch.cat([feat_bl, feat_br], dim=3) # -> (B, 1024, 14, 28)
    target_hr = torch.cat([top, bottom], dim=2)   # -> (B, 1024, 28, 28)
    
    return target_hr

In [ ]:



#  INITIALISATION DE LA PIPELINE



    # --- Initialisation des modèles ---
y = torch.randn(1, 202, 128, 128).to(dev)  # Exemple d'image HSI aléatoire pour l'initialisation
dofa,_,_,_ = models.create_dofa(y,y, path="dofaweights/")
dofa = dofa.to(dev)
spectral_earth = models.create_spectralearth(dev, path="spectral_earth/data")


if(train_7x14):
    naf = NAF(in_channels=128, kernel_size=7).to(dev)
else:
    naf = NAF(in_channels=128).to(dev)
    
    # Geler DOFA et Spectral Earth
dofa.eval()
for param in dofa.parameters():
    param.requires_grad = False
        
spectral_earth.eval()
for param in spectral_earth.parameters():
    param.requires_grad = False

    # NAF est le seul modèle en mode entraînement
naf.train()
    

criterion = torch_nn.MSELoss()
optimizer = AdamW(naf.parameters(), lr=5e-5)
    
epochs = 20 
 

In [ ]:
#Recuperer les longueurs d'ondes de l'image hyperspectrale

import pandas as pd


chemin_csv = "enmap_bands_remapped.csv"
df = pd.read_csv(chemin_csv)


wavelengths = df['center_nm'].tolist()


print(f"Nombre de longueurs d'onde récupérées : {len(wavelengths)}")
print(f"Premières valeurs : {wavelengths[:5]}")

Nombre de longueurs d'onde récupérées : 202
Premières valeurs : [418.24, 423.874, 429.294, 434.528, 439.603]


In [ ]:
import os
import glob
import torch
import rasterio
from torch.utils.data import Dataset, DataLoader

class EnmapDataset(Dataset):
    def __init__(self, folder_path, max_images=None): # Ajout de max_images
        self.folder_path = folder_path
        # Récupère tous les fichiers
        all_image_paths = glob.glob(os.path.join(folder_path, "*.tif"))
        
        # Tronquer la liste si max_images est spécifié
        if max_images is not None:
            self.image_paths = all_image_paths[:max_images]
            print(f"Utilisation de {len(self.image_paths)} images sur {len(all_image_paths)} disponibles.")
        else:
            self.image_paths = all_image_paths
            print(f"Utilisation de toutes les {len(self.image_paths)} images.")
        


    def __len__(self):
        """Retourne le nombre total d'images."""
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Charge, convertit et normalise une image spécifique.
        """
        img_path = self.image_paths[idx]
        
        with rasterio.open(img_path) as img:
            array = img.read()
            

        # On convertit en float et on normalise (pour ramener les valeurs autour de 0-1)
        tensor_img = torch.from_numpy(array).float()
        tensor_img = tensor_img / tensor_img.max() 
        
        return tensor_img



enmap_folder = "enmap"
dataset = EnmapDataset(folder_path=enmap_folder, max_images=600)

g = torch.Generator(device=dev) #Pour eviter les problemes de differents devices entre le DataLoader et le Dataset


dataloader = DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=True, 
    num_workers=2, 
    generator=g  
)

# Petit test rapide pour vérifier que tout fonctionne
for batch_hsi in dataloader:
    print(f"Format d'un batch sortant du dataloader : {batch_hsi.shape}")
    break 

KeyboardInterrupt: 

In [ ]:
#4 . Boucle d'entraînement avec crop (utilisation de l'entrainement avec crop pour les images 128x128)
print("Début de l'entraînement...")

best_loss = float('inf') 
save_path = "naf_hyperspectral_best.pth"

for epoch in range(epochs):
    epoch_losses = [] 
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
    
    for batch_idx, batch_hsi in pbar:
        
        batch_hsi = batch_hsi.to(dev) 
        
        optimizer.zero_grad()
        
        #  Générer la Vérité Terrain (Target) 
        target_hr = get_target_hr_features(dofa, batch_hsi, wavelengths)
        
        #  Générer les Features LR (Input) 
        hsi_224 = F.interpolate(batch_hsi, size=(224, 224), mode='bilinear', align_corners=False)
        with torch.no_grad():
            feat_lr_raw = models.extract_features(dofa, hsi_224, wavelengths)
            feat_lr = reshape_dofa(feat_lr_raw)
            
        #  Générer le Guidage HR 
        with torch.no_grad():
            guidance_hr = models.spectral_adapter(spectral_earth, batch_hsi)

        #  Forward NAF
        pred_hr = naf(guidance_hr, feat_lr, output_size=(28, 28)) 
        
        # Loss & Backprop 
        loss = criterion(pred_hr, target_hr)
        loss.backward()
        optimizer.step()

        
        epoch_losses.append(loss.item())
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    # Calcul de la loss moyenne de l'époque
    avg_epoch_loss = sum(epoch_losses) / len(epoch_losses)
    # Sauvegarde conditionnelle
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': naf.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, save_path)
        print(f"\nNouveau meilleur modèle sauvegardé avec une loss de: {best_loss:.4f}")
    else:
        print(f"\nLoss de cette époque ({avg_epoch_loss:.4f}) supérieure au meilleur score ({best_loss:.4f}), pas de sauvegarde.")

Début de l'entraînement...


Epoch 1/20: 100%|██████████| 600/600 [04:26<00:00,  2.25batch/s, Loss=2.9908] 


Nouveau meilleur modèle sauvegardé avec une loss de: 4.6604



Epoch 2/20: 100%|██████████| 600/600 [04:25<00:00,  2.26batch/s, Loss=3.4212] 


Nouveau meilleur modèle sauvegardé avec une loss de: 4.3186


...


Loss de cette époque (4.1135) supérieure au meilleur score (4.1124), pas de sauvegarde.



Epoch 19/20: 100%|██████████| 600/600 [04:32<00:00,  2.20batch/s, Loss=3.6652] 


Loss de cette époque (4.1126) supérieure au meilleur score (4.1124), pas de sauvegarde.



Epoch 20/20: 100%|██████████| 600/600 [04:35<00:00,  2.18batch/s, Loss=3.2706] 


Nouveau meilleur modèle sauvegardé avec une loss de: 4.1071


In [ ]:

    # SAUVEGARDE DES POIDS

save_path = "naf_hyperspectral_weights.pth"
torch.save({
        'epoch': epochs,
        'model_state_dict': naf.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss.item(),
    }, save_path)
    
print(f"Entraînement terminé. Poids sauvegardés dans {save_path}")



Entraînement terminé. Poids sauvegardés dans naf_hyperspectral_weights.pth


In [ ]:
#Train naf classic 7x7 -> 14x14

from tqdm import tqdm
import torch.nn.functional as F

print("Début de l'entraînement (Méthode de sous-échantillonnage 7x7 -> 14x14)...")

best_loss = float('inf')
save_path = "naf_hyperspectral_best_7x14.pth"

for epoch in range(epochs):
    epoch_losses = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
    
    for batch_idx, batch_hsi in pbar:
        batch_hsi = batch_hsi.to(dev) 
        optimizer.zero_grad()
        

        hsi_224 = F.interpolate(batch_hsi, size=(224, 224), mode='bilinear', align_corners=False)
        with torch.no_grad():
            target_hr_raw = models.extract_features(dofa, hsi_224, wavelengths)
            target_hr = reshape_dofa(target_hr_raw) # -> (Batch, 1024, 14, 14)
            

            feat_lr = F.interpolate(target_hr, size=(7, 7), mode='bilinear', align_corners=False)
            
            
            guidance_hr = models.spectral_adapter(spectral_earth, batch_hsi)

        
        pred_hr = naf(guidance_hr, feat_lr, output_size=(14, 14)) 
        
       
        loss = criterion(pred_hr, target_hr) # On compare du 14x14 avec du 14x14
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(naf.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_losses.append(loss.item())
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
        
        
    avg_epoch_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"Epoch {epoch+1}/{epochs} - Loss moyenne: {avg_epoch_loss:.4f}")
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        torch.save({'model_state_dict': naf.state_dict()}, save_path)

Début de l'entraînement (Méthode de sous-échantillonnage 7x7 -> 14x14)...


Epoch 1/20: 100%|██████████| 600/600 [01:12<00:00,  8.26batch/s, Loss=1.1529]

Epoch 1/20 - Loss moyenne: 1.7657



Epoch 2/20: 100%|██████████| 600/600 [01:08<00:00,  8.80batch/s, Loss=0.8520]

Epoch 2/20 - Loss moyenne: 1.5225


...

Epoch 18/20 - Loss moyenne: 1.3008


Epoch 19/20: 100%|██████████| 600/600 [01:14<00:00,  8.10batch/s, Loss=1.0872]

Epoch 19/20 - Loss moyenne: 1.2992



Epoch 20/20: 100%|██████████| 600/600 [01:11<00:00,  8.36batch/s, Loss=1.0033]

Epoch 20/20 - Loss moyenne: 1.2980
